In [ ]:
from dataclasses import dataclass
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import zscore, pearsonr
from skfda import FDataGrid
from skfda.preprocessing.dim_reduction import FPCA
from skfda.misc.regularization import L2Regularization
from skfda.misc.operators import LinearDifferentialOperator
from skfda.exploratory.visualization import FPCAPlot
from skfda.representation.basis import BSplineBasis, FDataBasis
from skfda.preprocessing.smoothing import BasisSmoother

import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import skfda

def prepare_error_learning(subject_list):
    behavioural_ids = subject_list["dicom_dir"].to_list()
    subid_map = dict(zip(subject_list["dicom_dir"], subject_list["participant_id"]))

    error_learning = pd.read_csv("../data/behavioural-data/ml_trial_data.csv")
    error_learning["Error"] = np.degrees(error_learning["Error"])
    error_learning = error_learning[error_learning["Subject"].isin(behavioural_ids)]
    error_learning["subid"] = error_learning["Subject"].map(subid_map)
    error_learning = error_learning[error_learning["Block"] == 3]
    error_learning = error_learning.sort_values(["subid", "Trial"])
    error_learning["Error"] = 45 - error_learning["Error"]
    el_long = error_learning.assign(Learning=lambda df: df["Error"])[["subid", "Trial", "Learning"]]

    return el_long

def prepare_reinforcement_learning(subject_list):
    behavioural_ids = subject_list["dicom_dir"].to_list()
    subid_map = dict(zip(subject_list["dicom_dir"], subject_list["participant_id"]))

    rl = pd.read_csv("../data/behavioural-data/rl_trial_data.csv")
    rl = rl[rl["Subject"].isin(behavioural_ids)]
    rl["subid"] = rl["Subject"].map(subid_map)
    rl = rl[rl["Block"] == 2]
    rl = rl.sort_values(["subid", "Trial"])

    first_80 = rl.groupby("subid").head(80)
    last_80  = rl.groupby("subid").tail(80)

    rl_sel = pd.concat([first_80, last_80]).sort_values(["subid", "Trial"]).copy()
    rl_sel["Trial"] = rl_sel.groupby("subid").cumcount() + 1
    rl_long = rl_sel.assign(Learning=lambda df: df["Score"])[["subid", "Trial", "Learning"]]
    return rl_long

def build_combined_long(el_long, rl_long):
    el_long = el_long.copy()
    rl_long = rl_long.copy()
    n_el_trials = int(el_long["Trial"].max())
    el_long["CombinedTrial"] = el_long["Trial"]
    rl_long["CombinedTrial"] = rl_long["Trial"] + n_el_trials
    combined_long = pd.concat([el_long[["subid", "CombinedTrial", "Learning"]],
                    rl_long[["subid", "CombinedTrial", "Learning"]]], ignore_index=True)
    return combined_long, n_el_trials

def fill_data(df):
    df = df.copy()
    df = df.sort_values(["subid", "CombinedTrial"])
    wide = df.pivot_table(index="subid", columns="CombinedTrial", values="Learning", aggfunc="median")
    wide.columns = pd.to_numeric(wide.columns)
    wide = wide.sort_index(axis=1)
    wide = wide.astype(float)
    wide = wide.interpolate(axis=1, limit_direction="both")
    wide = wide.bfill(axis=1).ffill(axis=1)
    return wide

def median_presmooth_wide(wide, window = 5):
    def _row_med(row: pd.Series) -> pd.Series:
        return row.rolling(window=window, center=True, min_periods=1).median()
    wide_smooth = wide.apply(_row_med, axis=1)
    return wide_smooth

def smooth_with_splines_gcv(wide, n_basis = None, lambdas = None):
    grid_points = wide.columns.values.astype(float, copy=False)
    data_matrix = wide.values         
    n_grid = len(grid_points)

    if n_basis is None:
        n_basis = min(n_grid, 40)
    if lambdas is None:
        lambdas = np.logspace(-4, 3, 20)

    fd = FDataGrid(data_matrix=data_matrix, grid_points=grid_points)
    basis = BSplineBasis(domain_range=(grid_points[0], grid_points[-1]), n_basis=n_basis, order=4)
    differential_op = LinearDifferentialOperator(2)
    regularization = L2Regularization(differential_op)

    best_score = -np.inf
    best_fd_basis = None
    best_lambda = None

    raw = np.asarray(fd.data_matrix).squeeze()

    for lam in lambdas:
        smoother = BasisSmoother(basis=basis, smoothing_parameter=float(lam), regularization=regularization, return_basis=True)
        fd_basis = smoother.fit_transform(fd) 
        smooth = fd_basis(grid_points).squeeze()
        resid = raw - smooth
        ss_res = np.sum(resid ** 2)
        ss_tot = np.sum((raw - raw.mean()) ** 2)
        score = 1.0 - ss_res / ss_tot
        if score > best_score:
            best_score = score
            best_fd_basis = fd_basis
            best_lambda = lam

    return best_fd_basis, float(best_lambda)
    
@dataclass
class FPCAResult:
    fpca: FPCA
    basis_fd: skfda.representation.basis.FDataBasis
    scores: pd.DataFrame

def run_fpca(basis_fd: skfda.representation.basis.FDataBasis, subids: list[str], n_components: int = 2) -> FPCAResult:
    fpca = FPCA(n_components=n_components, components_basis=basis_fd.basis, regularization=None)
    scores = fpca.fit_transform(basis_fd)  
    score_cols = [f"FPC{i+1}" for i in range(n_components)]
    scores_df = pd.DataFrame(scores, columns=score_cols)
    scores_df.insert(0, "subid", subids)

    print("Explained variance ratio:", fpca.explained_variance_ratio_)
    return FPCAResult(fpca=fpca, basis_fd=basis_fd, scores=scores_df)

def plot_fpc_deformation(basis_fd: skfda.representation.basis.FDataBasis, fpca: FPCA, component = 0, xlabel = "Trial",
                         ylabel = "Learning", title = None, t_min = None, t_max = None):
    t0, t1 = basis_fd.domain_range[0]
    t = np.linspace(t0, t1, 400)

    mean_vals = basis_fd.mean()(t).squeeze()
    comp_vals = fpca.components_[component](t).squeeze()

    if (t_min is not None) or (t_max is not None):
        mask = np.ones_like(t, dtype=bool)
        if t_min is not None:
            mask &= t >= t_min
        if t_max is not None:
            mask &= t <= t_max
        t = t[mask]
        mean_vals = mean_vals[mask]
        comp_vals = comp_vals[mask]

    ev = getattr(fpca, "explained_variance_", None)
    scale = float(np.sqrt(ev[component])) if ev is not None else 1.0

    neg_factors = np.array([0.4, 0.8, 1.2])
    pos_factors = np.array([0.4, 0.8, 1.2])

    red_colors = ["#BBCDF5", "#85A0F3", "#4251BD"]
    green_colors = ["#E6D4C9", "#E69B7E", "#BF4F43"]

    fig, ax = plt.subplots(figsize=(12, 6))

    last_curve = mean_vals.copy()
    for factor, color in zip(neg_factors, red_colors):
        this_curve = mean_vals - factor * scale * comp_vals
        ax.fill_between(t, this_curve, last_curve, color=color, alpha=1)
        last_curve = this_curve

    last_curve = mean_vals.copy()
    for factor, color in zip(pos_factors, green_colors):
        this_curve = mean_vals + factor * scale * comp_vals
        ax.fill_between(t, last_curve, this_curve, color=color, alpha=1)
        last_curve = this_curve

    ax.plot(t, mean_vals, color="black", linewidth=3)

    ax.set_xlabel(xlabel, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    if title is not None:
        ax.set_title(title, fontsize=14, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xticks(np.arange(int(t.min()), int(t.max()) + 1, 1))
    ax.set_xticklabels([""] * len(ax.get_xticks()))
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["right"].set_linewidth(1.5)
    plt.tight_layout()
    plt.savefig(f"../figures/fPCA/{title}.png", dpi=800, transparent=True)
    plt.close()
    
def plot_extreme_learning_curves(data_long, scores_df, out_prefix, segment_name, value_col = "Learning", 
                                 subject_col = "subid", trial_col = "CombinedTrial", smooth_window = 5):
    neg_colors = ["#BBCDF5", "#85A0F3", "#4251BD"]
    pos_colors = ["#E6D4C9", "#E69B7E", "#BF4F43"]
    six_colors = neg_colors[::-1] + pos_colors
    fpc_cmap = LinearSegmentedColormap.from_list("fpc_diverging", six_colors)

    df_scores = scores_df.set_index(subject_col)
    sorted_ids = df_scores["FPC1"].sort_values()

    low_id = sorted_ids.index[0]
    high_id = sorted_ids.index[-1]

    print(segment_name, low_id, high_id)

    low_color  = fpc_cmap(0.0)   
    high_color = fpc_cmap(1.0)  

    def smooth_curve(sub_df):
        sub_df = sub_df.sort_values(trial_col).copy()
        sub_df["smoothed"] = (sub_df[value_col].rolling(window=smooth_window, center=True, min_periods=1).median())
        return sub_df

    fig, ax = plt.subplots(figsize=(8, 5))

    for subject in data_long[subject_col].unique():
        if subject in [low_id, high_id]:
            continue

        sub = data_long[data_long[subject_col] == subject]
        sub = smooth_curve(sub)

        ax.plot(sub[trial_col], sub["smoothed"], color="lightgrey", linewidth=1, alpha=0.4, zorder=1)

    sub_low = data_long[data_long[subject_col] == low_id]
    sub_low = smooth_curve(sub_low)
    ax.plot(sub_low[trial_col], sub_low["smoothed"], color=low_color, linewidth=3, zorder=10)

    sub_high = data_long[data_long[subject_col] == high_id]
    sub_high = smooth_curve(sub_high)
    ax.plot(sub_high[trial_col], sub_high["smoothed"], color=high_color, linewidth=3, zorder=10)

    trials = np.sort(data_long[trial_col].unique())
    ax.set_xticks(trials)

    if len(trials) > 30:
        step = max(1, len(trials) // 20)
        ax.set_xticklabels([str(t) if (i % step == 0) else "" for i, t in enumerate(trials)])
    else:
        ax.set_xticklabels(trials)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.savefig(f"../figures/fPCA/{segment_name}_indiv_learning.png", dpi=800, transparent=True)
    plt.close(fig)

def plot_extreme_learning_curves_fpc2(data_long, scores_df, out_prefix, segment_name, value_col = "Learning", 
                                 subject_col = "subid", trial_col = "CombinedTrial", smooth_window = 5):
    neg_colors = ["#BBCDF5", "#85A0F3", "#4251BD"]
    pos_colors = ["#E6D4C9", "#E69B7E", "#BF4F43"]
    six_colors = neg_colors[::-1] + pos_colors
    fpc_cmap = LinearSegmentedColormap.from_list("fpc_diverging", six_colors)

    df_scores = scores_df.set_index(subject_col)
    sorted_ids = df_scores["FPC2"].sort_values()

    low_id = sorted_ids.index[0]
    high_id = sorted_ids.index[-1]

    print(segment_name, low_id, high_id)

    low_color  = fpc_cmap(0.0)   
    high_color = fpc_cmap(1.0)  

    def smooth_curve(sub_df):
        sub_df = sub_df.sort_values(trial_col).copy()
        sub_df["smoothed"] = (sub_df[value_col].rolling(window=smooth_window, center=True, min_periods=1).median())
        return sub_df

    fig, ax = plt.subplots(figsize=(8, 5))

    for subject in data_long[subject_col].unique():
        if subject in [low_id, high_id]:
            continue

        sub = data_long[data_long[subject_col] == subject]
        sub = smooth_curve(sub)

        ax.plot(sub[trial_col], sub["smoothed"], color="lightgrey", linewidth=1, alpha=0.4, zorder=1)

    sub_low = data_long[data_long[subject_col] == low_id]
    sub_low = smooth_curve(sub_low)
    ax.plot(sub_low[trial_col], sub_low["smoothed"], color=low_color, linewidth=3, zorder=10)

    sub_high = data_long[data_long[subject_col] == high_id]
    sub_high = smooth_curve(sub_high)
    ax.plot(sub_high[trial_col], sub_high["smoothed"], color=high_color, linewidth=3, zorder=10)

    trials = np.sort(data_long[trial_col].unique())
    ax.set_xticks(trials)

    if len(trials) > 30:
        step = max(1, len(trials) // 20)
        ax.set_xticklabels([str(t) if (i % step == 0) else "" for i, t in enumerate(trials)])
    else:
        ax.set_xticklabels(trials)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.savefig(f"../figures/fPCA/{segment_name}_indiv_learning_fpc2.png", dpi=800, transparent=True)
    plt.close(fig)

def run_fpca(basis_fd: skfda.representation.basis.FDataBasis, subids: list[str], n_components: int = 2):
    fpca = FPCA(n_components=n_components, components_basis=basis_fd.basis, regularization=None)
    scores = fpca.fit_transform(basis_fd)  
    score_cols = [f"FPC{i+1}" for i in range(n_components)]
    scores_df = pd.DataFrame(scores, columns=score_cols)
    scores_df.insert(0, "subid", subids)

    print("Explained variance ratio:", fpca.explained_variance_ratio_)
    return FPCAResult(fpca=fpca, basis_fd=basis_fd, scores=scores_df)


def main():
    os.makedirs("../figures/fPCA", exist_ok=True)
    
    #1. Load subject list and build IDs
    subject_list = pd.read_csv("../data/behavioural-data/participants.tsv", sep="\t")
    subject_list = subject_list[subject_list["exclude"] == 0]
    subject_list["participant_id"] = ( "sub-" + subject_list["participant_id"].astype(str).str.zfill(2))

    subject_colors = np.load("../figures/subject_colours.npy")
    subid_list = subject_list["participant_id"].tolist()
    sub_to_color = dict(zip(subid_list, subject_colors))

    neg_colors = ["#BBCDF5", "#85A0F3", "#4251BD"]
    pos_colors = ["#E6D4C9", "#E69B7E", "#BF4F43"]
    six_colors = neg_colors[::-1] + pos_colors
    fpc_cmap = LinearSegmentedColormap.from_list("fpc_diverging", six_colors)

    #2. Retrive EL and RL data
    el_long = prepare_error_learning(subject_list)
    rl_long = prepare_reinforcement_learning(subject_list)

    combined_long, n_el_trials = build_combined_long(el_long, rl_long)
    wide_combined = fill_data(combined_long)
    wide_med = median_presmooth_wide(wide_combined, window=5)
        
    basis_fd_smooth, best_lambda = smooth_with_splines_gcv(wide_med,n_basis=15)

    #3. Run FPCA
    subids = list(wide_combined.index)
    fpca_result = run_fpca(basis_fd_smooth, subids=subids, n_components=2)
    
    t0, t1 = basis_fd_smooth.domain_range[0]
    el_t_min = t0
    el_t_max = float(n_el_trials)
    rl_t_min = float(n_el_trials + 1)
    rl_t_max = t1

    plot_fpc_deformation(basis_fd=fpca_result.basis_fd, fpca=fpca_result.fpca, component=0, t_min=el_t_min, 
                         t_max=el_t_max, xlabel="EL trial", ylabel="Learning (Error / Score)", title="FPC1_EL")
    plot_fpc_deformation(basis_fd=fpca_result.basis_fd, fpca=fpca_result.fpca, component=0, t_min=rl_t_min,
                         t_max=rl_t_max, xlabel="RL trial", ylabel="Learning (Error / Score)", title="FPC1_RL")

    plot_fpc_deformation(basis_fd=fpca_result.basis_fd, fpca=fpca_result.fpca, component=1, t_min=el_t_min, 
                         t_max=el_t_max, xlabel="EL trial", ylabel="Learning", title="FPC2_EL")
    plot_fpc_deformation(basis_fd=fpca_result.basis_fd, fpca=fpca_result.fpca, component=1, t_min=rl_t_min,
                         t_max=rl_t_max, xlabel="RL trial", ylabel="Learning", title="FPC2_RL")

    FPCA_data = fpca_result.scores
    FPCA_data["FPC1_z"] = zscore(FPCA_data["FPC1"])
    FPCA_data["FPC2_z"] = zscore(FPCA_data["FPC2"])
    FPCA_data["colour"] = FPCA_data["subid"].map(sub_to_color)
    FPCA_data.to_csv("../output/fpca_scores.csv")

    EL_segment = combined_long[combined_long["CombinedTrial"] <= n_el_trials]
    RL_segment = combined_long[combined_long["CombinedTrial"] > n_el_trials]

    plot_extreme_learning_curves(data_long=EL_segment, scores_df=FPCA_data, out_prefix="../figures/fPCA/extremes",
        segment_name="EL", value_col="Learning", subject_col="subid", trial_col="CombinedTrial", smooth_window=5)
    plot_extreme_learning_curves(data_long=RL_segment, scores_df=FPCA_data, out_prefix="../figures/fPCA/extremes",
        segment_name="RL", value_col="Learning", subject_col="subid", trial_col="CombinedTrial", smooth_window=5)

    np.random.seed(12345)
    jitter = np.random.uniform(-0.005, 0.005, len(FPCA_data))
    
    plt.figure(figsize=(1.5, 3))
    plt.scatter(jitter, FPCA_data["FPC1_z"], c=FPCA_data["colour"], edgecolor="black", s=100)
    plt.xlim(-0.01, 0.01)  
    plt.xticks([])
    plt.xlabel("")
    plt.tight_layout()
    plt.savefig("../figures/fPCA/sub_loadings.png", dpi=800, transparent=True)
    plt.close()

    norm = (FPCA_data["FPC1"] - FPCA_data["FPC1"].min()) / (FPCA_data["FPC1"].max() - FPCA_data["FPC1"].min())
    colors = norm.apply(fpc_cmap)

    plt.figure(figsize=(1.5, 3))
    plt.scatter(jitter, FPCA_data["FPC1_z"], c=colors, edgecolor="black", s=100)
    plt.xlim(-0.01, 0.01)  
    plt.xticks([])
    plt.xlabel("")
    plt.tight_layout()
    plt.savefig("../figures/fPCA/sub_loadings_fpc1_cmap.png", dpi=800, transparent=True)
    plt.close()

    plot_extreme_learning_curves_fpc2(data_long=EL_segment, scores_df=FPCA_data, out_prefix="../figures/fPCA/extremes",
        segment_name="EL", value_col="Learning", subject_col="subid", trial_col="CombinedTrial", smooth_window=5)
    plot_extreme_learning_curves_fpc2(data_long=RL_segment, scores_df=FPCA_data, out_prefix="../figures/fPCA/extremes",
        segment_name="RL", value_col="Learning", subject_col="subid", trial_col="CombinedTrial", smooth_window=5)
    
    norm = (FPCA_data["FPC2"] - FPCA_data["FPC2"].min()) / (FPCA_data["FPC2"].max() - FPCA_data["FPC2"].min())
    colors = norm.apply(fpc_cmap)
    
    plt.figure(figsize=(1.5, 3))
    plt.scatter(jitter, FPCA_data["FPC2_z"], c=colors, edgecolor="black", s=100)
    plt.xlim(-0.01, 0.01)  
    plt.xticks([])
    plt.xlabel("")
    plt.tight_layout()
    plt.savefig("../figures/fPCA/sub_loadings_fpc2_cmap.png", dpi=800, transparent=True)
    plt.close()
    
main()